# Conversations as State

Each call to an LLM API is **stateless**. 

Thus, we keep **state** using conversations:

<img src="assets/08_llm_apis.png" alt="conversation" width="550"/>

* **system**: initial instructions
* **user**: your instructions
* **assistant**: the LLM response

The first message is `system`, then pairs of `user` and `assistant`. 


In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',
)

messages = [
    {"role": "system", "content": "You are a helpful assistant. Keep responses concise."},
]

while True:
    user_input = input("YOU: ").strip()
    if not user_input:
        continue

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(model="gemma4:e2b", messages=messages)
    assistant_reply = response.choices[0].message.content.strip()
    messages.append({"role": "assistant", "content": assistant_reply})

    print(f"\nASSISTANT: {assistant_reply}")
    print("\n" + "-" * 76)
    print("Messages state:")
    print(json.dumps(messages, indent=2))
    print("-" * 76 + "\n")

# Little number guessing game

In [ ]:
secret = int(input("Think of a number between 1 and 1000 (don't tell me): "))
assert(1 <= secret <= 1000)

messages = [
    {"role": "system",
     "content": "You are playing a number guessing game. The user has picked a number between 1 and 1000. Guess it as best as you can. Ask only one number per turn. After each guess the user will tell you 'higher', 'lower', or 'correct'. Keep responses super short: just state your guess."
    },
]

for _ in range(100):
    response = client.chat.completions.create(
        model="gemma4:e2b",
        messages=messages
    )
    assistant_guess = response.choices[0].message.content.strip()
    messages.append({"role": "assistant", "content": assistant_guess})
    

    # auto-reply based on the secret number
    guessed = int(''.join(filter(str.isdigit, assistant_guess.split()[0] if assistant_guess else '0')) or 0)
    if guessed == secret:
        user_content = "Correct! You got it!"
        print(f"USER: {user_content}")
        messages.append({"role": "user", "content": user_content})
        break
    elif guessed < secret:
        user_content = "Higher."
    else:
        user_content = "Lower."

    print("-" * 76)
    print(f"\nASSISTANT: {assistant_guess}")
    print(f"USER: {user_content}\n\n")
    messages.append({"role": "user", "content": user_content})
    print(json.dumps(messages, indent=2) + "\n")